In [1]:
import os
import sys
sys.path.append(os.path.expanduser('~'))
import numpy as np 
from matplotlib import pyplot as plt
from datetime import datetime as dt
import figures as fgs
from albatros_analysis.src.correlations import baseband_data_classes as bdc
from albatros_analysis.src.utils import baseband_utils as butils
from albatros_analysis.src.utils import orbcomm_utils as outils
import numba as nb
import time
import importlib
import json
from pyuvdata import UVData

BDC is using numpy
BDC is using numpy


In [3]:
config_path = '/home/thomasb/albatros_analysis/scripts/xcorr/config/config_bright_test.json'
disk_path = 'XXXXX'

In [4]:
#open config and get metadata
with open(config_path, "r") as f:
    config = json.load(f)
ant_names, ant_coords, dir_parents, spec_offsets = [], [], [], []
# Call get_starting_index for all antennas except reference
for i, (ant, details) in enumerate(config["antennas"].items()):
    dir_parents.append(details["path"])
    spec_offsets.append(details["clock_offset"])
    ant_names.append(details["name"])
    ant_coords.append(details["coordinates"])

ant_numbers = np.array([int(name.split()[1]) for name in ant_names])
batch_start_ts = config["correlation"]["start_timestamp"]
batch_end_ts = config["correlation"]["end_timestamp"]
chanstart = config["frequency"]["start_channel"]
chanend = config["frequency"]["end_channel"]
osamp = config["correlation"]["osamp"]
pfb_size = config["correlation"]["pfb_size"]
new_acclen = config["correlation"]["new_acclen"]

new_chunk_len = (new_acclen*4096*osamp)/250e6
tle_path = outils.get_tle_file(batch_start_ts, "/project/rrg-sievers/mohanagr/OCOMM_TLES")
print(new_acclen)

512


In [ ]:
def objective(time_offset,
              data,
              freq,
              t_start,
              t_end,
              satID,
              osamp,
              antnums,
              ant_coords,
              baseband_spectrum_T = 4096/250e6,
              acclen = 1024,
              plot=False):
    

    T_SPECTRA = baseband_spectrum_T * osamp
    tle_path = outils.get_tle_file(t_start, "/project/rrg-sievers/mohanagr/OCOMM_TLES")
    #ant_to_use = [0,2,3,4,5,6]
    chisq = 0
    # time_offset = time_offset[0]
    sum_wt = 0
    for i in range(len(antnums)):
        for j in range(i+1, len(antnums)):
            ai_num = antnums[i]
            aj_num = antnums[j]
            bl_slice = uv.antpair2ind(ai_num, aj_num)
            bl_data = data_all[bl_slice, :, :] #pol XX for now
            a1_coords = ant_coords[i]
            a2_coords = ant_coords[j]
            dist = haversine(a1_coords,a2_coords)
            sum_wt += dist**2
            dly = outils.get_sat_delay(
                                a1_coords,
                                a2_coords,
                                tle_path,
                                t_start+time_offset,
                                int(t_end-t_start)+1,
                                satID,
                                altaz=False
                            )
            delay = np.interp(
                np.arange(0, data.shape[2]) * T_SPECTRA, np.arange(0, int(t_end-t_start)+1), dly
            )

            
            spec2_phased = np.empty_like(data[aj_,0,:])
            spec2_phased = apply_delay_1d(data[aj,0,:], spec2_phased, -delay, freq)
            Vxx = xcorr_avg_1d(data_slice[ai,0,:],spec2_phased,acclen)
            spec2_phased = apply_delay_1d(data_slice[aj,1,:], spec2_phased, -delay, freq)
            Vyy = xcorr_avg_1d(data_slice[ai,1,:],spec2_phased,acclen)
            V=(Vxx+Vyy)/2
            if plot:
                plt.plot(np.unwrap(np.angle(V))-np.angle(V)[0],label=f'{ai}-{aj}')
                plt.axhline(6.28, ls='--',c='black')
                plt.axhline(-6.28,ls='--', c='black')
                plt.legend()
            Vnew = np.exp(1j*np.angle(V))
            chisq-= dist**2*np.abs(np.mean(Vnew[:165]))**2 #cut it off where snr drops
    chisq/=sum_wt
    
    print("offset", time_offset, "chisq", chisq)
    return chisq